# LAB3 — Bài 4 (trang 26): Gom cụm Titanic

**Sinh viên:** Trần Chí Nguyên — **MSSV:** 6451071052.

> Mỗi câu lập trình in họ tên và MSSV theo template. Chạy lần lượt từ trên xuống.

In [1]:
%matplotlib inline
HO_TEN = 'Trần Chí Nguyên'
MSSV = '6451071052'
def in_danh_tinh(): print(f'{HO_TEN} - {MSSV}')


## Câu 4. Gom cụm Titanic bằng k-Means và so sánh với nhãn sống sót

Tải Titanic CSV từ URL công khai, xử lý thiếu, mã hóa biến phân loại và chuẩn hóa thuộc tính. Chọn k=2. Vì nhãn cụm không tương ứng trực tiếp với nhãn survived, ánh xạ cụm sang lớp bằng Hungarian assignment trước khi tính confusion matrix/Precision/Recall/F1.

In [2]:
in_danh_tinh()
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import confusion_matrix, classification_report, adjusted_rand_score, silhouette_score
from scipy.optimize import linear_sum_assignment
TITANIC_URL='https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
df=pd.read_csv(TITANIC_URL)
print('Shape:',df.shape); display(df.head()); print('Missing values:\n',df.isna().sum())
label_col='Survived'; y=df[label_col].astype(int).to_numpy()
features=df[['Pclass','Sex','Age','SibSp','Parch','Fare','Embarked']].copy()
features['Age']=features['Age'].fillna(features['Age'].median())
features['Fare']=features['Fare'].fillna(features['Fare'].median())
features['Embarked']=features['Embarked'].fillna(features['Embarked'].mode().iloc[0])
features['Sex']=features['Sex'].fillna(features['Sex'].mode().iloc[0])
X=pd.get_dummies(features,columns=['Sex','Embarked'],dtype=float)
scaler=StandardScaler(); X_scaled=scaler.fit_transform(X)
kmeans=KMeans(n_clusters=2,random_state=42,n_init=20)
clusters=kmeans.fit_predict(X_scaled)
print('Silhouette:',silhouette_score(X_scaled,clusters)); print('ARI:',adjusted_rand_score(y,clusters))
# Tìm hoán vị nhãn cụm/lớp cho confusion matrix
cm_raw=confusion_matrix(y,clusters,labels=[0,1])
rows,cols=linear_sum_assignment(-cm_raw)
mapping={cluster:label for label,cluster in zip(rows,cols)}
y_cluster_aligned=np.array([mapping[c] for c in clusters])
cm=confusion_matrix(y,y_cluster_aligned,labels=[0,1])
print('Ánh xạ cụm -> survived:',mapping); print(classification_report(y,y_cluster_aligned,target_names=['Did not survive','Survived'],zero_division=0))
sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',xticklabels=['Did not survive','Survived'],yticklabels=['Did not survive','Survived'])
plt.xlabel('Cụm đã ánh xạ'); plt.ylabel('Nhãn thật'); plt.title('Titanic — Confusion matrix'); plt.show()
print(pd.crosstab(pd.Series(y,name='Survived'),pd.Series(clusters,name='Cluster')))

Trần Chí Nguyên - 6451071052


Shape: (891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


Missing values:
 PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64


Silhouette: 0.29948447663967237
ARI: 0.32782332017320776
Ánh xạ cụm -> survived: {np.int64(1): np.int64(0), np.int64(0): np.int64(1)}
                 precision    recall  f1-score   support

Did not survive       0.81      0.85      0.83       549
       Survived       0.74      0.69      0.71       342

       accuracy                           0.79       891
      macro avg       0.78      0.77      0.77       891
   weighted avg       0.79      0.79      0.79       891



Cluster     0    1
Survived          
0          82  467
1         235  107


## Nhận xét

So sánh hai cụm với Survived; nêu rõ k-Means là không giám sát, việc ánh xạ nhãn chỉ phục vụ so sánh. Confusion matrix và các độ đo này không biến bài toán thành phân lớp có giám sát.